# Remote GPU setup (SETUP-05)

Run this notebook on Colab for the GPU-heavy tasks (task 3 UVR, task 4 Food-101).
Before running: **Runtime > Change runtime type > GPU**. With the Google One
student plan, Colab gives more compute units than the free tier (better GPU
priority, e.g. T4/L4/A100 depending on availability, and longer sessions)
but sessions can still disconnect/time out, so checkpoints must live on a
persistent storage (Google Drive here), not Colab's local disk.

This notebook:
1. reports the GPU actually assigned to the session,
2. mounts Drive and clones the repo + data,
3. proves a training run can be interrupted and resumed without losing
   progress, using `common/checkpointing.py`.

## 1. GPU model available

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Persistent storage (Google Drive) for checkpoints

Colab's local disk is wiped when the session ends, so checkpoints go to
Drive instead. `drive.mount()` always prompts for *your own* Google
account -- Google requires that consent from whoever runs the notebook,
there's no way around it -- but by default it would give each teammate
their own private, unshared copy of the checkpoints under their own
`My Drive`.

To share the same checkpoints across the team instead (one-time setup):
1. One teammate creates a folder in their Drive, e.g. named
   `ml_tek5_checkpoints`, and shares it with the rest of the team
   (Editor access).
2. Every other teammate opens that shared folder in Drive, then
   right-click > **Organize > Add shortcut** > **My Drive**, keeping the
   default name so the path below matches for everyone.

If you'd rather keep private, per-teammate checkpoints, set
`TEAM_SHARED_CHECKPOINTS = False` below.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

TEAM_SHARED_CHECKPOINTS = True
TEAM_CHECKPOINT_FOLDER_NAME = "ml_tek5_checkpoints"

if TEAM_SHARED_CHECKPOINTS:
    CHECKPOINT_DIR = f"/content/drive/MyDrive/{TEAM_CHECKPOINT_FOLDER_NAME}"
else:
    CHECKPOINT_DIR = "/content/drive/MyDrive/ml_tek5_checkpoints_private"

existed_before = os.path.isdir(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if TEAM_SHARED_CHECKPOINTS and not existed_before:
    print(
        f"WARNING: '{CHECKPOINT_DIR}' didn't exist yet, so a new, "
        "*private* folder was just created in your own Drive -- it is "
        "NOT shared with your teammates. If you meant to use the shared "
        f"team folder, add a Drive shortcut named "
        f"'{TEAM_CHECKPOINT_FOLDER_NAME}' to your My Drive first (see "
        "section 2 above), then delete this folder and re-run this cell."
    )

print("Checkpoints:", CHECKPOINT_DIR)

## 3. Clone the repo and pull in the extra (GPU-task) dependencies

Colab already ships a CUDA-matched PyTorch build; reinstalling torch here
(e.g. via `uv sync`, which pins the CPU wheel for the no-GPU laptop) would
throw that away. Only the packages Colab doesn't already have are installed,
straight from `pyproject.toml`.

The repo is **private**, so cloning over HTTPS needs a GitHub token (a plain
`git clone` fails with `could not read Username ... No such device or
address`, since Colab can't prompt interactively for credentials). Create a
[fine-grained personal access token](https://github.com/settings/tokens)
with read-only `Contents` access to this repo, then either store it as a
Colab secret named `GITHUB_TOKEN` (recommended: toggle notebook access on)
or paste it when prompted below.

In [ ]:
import base64
import getpass

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if not token:
    token = getpass.getpass(
        "GitHub token (repo scope) for the private "
        "NattanCochet/G-AIA-900-PAR-9-1-machinelearning-11 repo: "
    )

REPO_URL = "https://github.com/NattanCochet/G-AIA-900-PAR-9-1-machinelearning-11.git"
REPO_DIR = "/content/ml-tek5-project"
# GitHub's git-over-HTTPS endpoint only accepts Basic auth (unlike the REST
# API, which also accepts Bearer) -- same scheme as actions/checkout. Passed
# as a one-off git config flag instead of embedding the token in the URL, so
# it never ends up persisted in {REPO_DIR}/.git/config.
basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH_HEADER = f"AUTHORIZATION: basic {basic_auth}"

# Clear any extraheader Colab (or a previous run) may already have
# configured for github.com -- combined with ours below it would send two
# "Authorization" headers, which GitHub rejects with 400 "Duplicate
# header: 'Authorization'". Also disable the credential helper so it can't
# inject a second one either.
!git config --global --unset-all http.https://github.com/.extraheader 2>/dev/null
!git config --system --unset-all http.https://github.com/.extraheader 2>/dev/null

if not os.path.isdir(REPO_DIR):
    !git -c credential.helper= -c http.extraheader="{AUTH_HEADER}" clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} config --local --unset-all http.https://github.com/.extraheader 2>/dev/null
    !git -C {REPO_DIR} -c credential.helper= -c http.extraheader="{AUTH_HEADER}" pull

In [ ]:
# Task 3 (UVR) + task 4 (Food-101) extras not already on Colab.
# Keep this list in sync with pyproject.toml's dependencies.
!pip install -q soundfile onnx onnxruntime onnx2pytorch mir_eval \
    transformers datasets accelerate timm

In [ ]:
import importlib
import sys

sys.path.insert(0, REPO_DIR)  # so `import common` resolves, as from the repo root
importlib.invalidate_caches()  # in case REPO_DIR was cached as empty by an earlier failed clone

import torch

assert torch.cuda.is_available(), "GPU not enabled: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 4. Checkpoint save/resume demo

Two ways to run this:
- **Quick check** (same session): run every cell below top to bottom.
  It proves the save/load logic is correct, but doesn't prove a real
  Colab disconnect is survived.
- **Real manual test** (recommended at least once): run "Shared demo
  setup" then "Part A", then **actually restart the runtime**
  (Runtime > Restart session, or for a harsher test, Runtime >
  Disconnect and delete runtime, which wipes the VM entirely), then
  re-run sections 1-3 above (GPU check, Drive mount, clone+install)
  plus "Shared demo setup", and finally run only "Part B" (not Part A
  again). Only the checkpoint and the reference-losses file on Drive
  survive a full disconnect; Part B loads both from there, so if it
  still matches you've proven the setup survives a real session loss,
  not just a simulated one.

`ModuleNotFoundError: No module named 'common'` below means section 3
(clone + `sys.path`) wasn't (re-)run in this runtime; "Shared demo
setup" re-adds the repo to `sys.path` itself as a fallback, but the repo
still has to actually be cloned to `/content/ml-tek5-project` first.

### Shared demo setup (re-run this after restarting too)

In [ ]:
import importlib
import sys

# Fallback in case this cell runs before section 3's sys.path.insert,
# e.g. right after a runtime restart.
REPO_DIR = globals().get("REPO_DIR", "/content/ml-tek5-project")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()  # in case REPO_DIR was cached as empty by an earlier failed clone

import torch
from torch import nn

from common import set_seed
from common.checkpointing import load_checkpoint, save_checkpoint

device = torch.device("cuda")
ckpt_path = f"{CHECKPOINT_DIR}/demo_checkpoint.pt"
reference_path = f"{CHECKPOINT_DIR}/demo_reference_losses.json"


def make_model_and_opt():
    set_seed(0)
    model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    return model, opt


set_seed(0)
x = torch.randn(64, 10, device=device)
y = torch.randn(64, 1, device=device)
loss_fn = nn.MSELoss()


def run(model, opt, start_epoch, end_epoch, save=False):
    losses = []
    for epoch in range(start_epoch, end_epoch):
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if save:
            save_checkpoint(ckpt_path, model, opt, epoch)
    return losses

### Part A — run now

Computes the uninterrupted reference run, then trains epochs 0-4
while checkpointing every epoch, and saves both to Drive so Part B
can find them even after a real restart.

In [ ]:
import json

model_a, opt_a = make_model_and_opt()
straight_losses = run(model_a, opt_a, 0, 10)

model_b, opt_b = make_model_and_opt()
first_half = run(model_b, opt_b, 0, 5, save=True)

with open(reference_path, "w") as f:
    json.dump({"straight": straight_losses, "first_half": first_half}, f)

print("straight  :", [round(v, 6) for v in straight_losses])
print("first_half:", [round(v, 6) for v in first_half])
print("\nNow restart the runtime (Runtime menu), then re-run cells")
print("1-3 and 'Shared demo setup' above, and run Part B below.")

### Part B — run after restarting the runtime

Do **not** re-run Part A first (that would start training over from
epoch 0 and defeat the test). This loads the checkpoint and the
reference losses purely from Drive, as if this were a brand new
session, and continues training from epoch 5.

In [ ]:
import json

with open(reference_path) as f:
    reference = json.load(f)

model_c, opt_c = make_model_and_opt()
next_epoch, _ = load_checkpoint(ckpt_path, model_c, opt_c, map_location=device)
assert next_epoch == 5, next_epoch
second_half = run(model_c, opt_c, next_epoch, 10)

resumed_losses = reference["first_half"] + second_half
print("straight:", [round(v, 6) for v in reference["straight"]])
print("resumed :", [round(v, 6) for v in resumed_losses])
assert resumed_losses == reference["straight"], "resume lost progress / diverged!"
print("OK: training resumed after the restart with zero lost progress")

## 5. Using this in a real task 3 / task 4 training script

```python
from common.checkpointing import save_checkpoint, load_checkpoint

start_epoch = 0
if os.path.exists(ckpt_path):
    start_epoch, extra = load_checkpoint(ckpt_path, model, optimizer, map_location=device)

for epoch in range(start_epoch, total_epochs):
    train_one_epoch(...)
    save_checkpoint(ckpt_path, model, optimizer, epoch)
```

Keep `ckpt_path` under `CHECKPOINT_DIR` (Drive), and re-run this notebook's
clone+install cells from the top the next time the session disconnects; the
training loop will pick up where it left off.

## 6. Run task 4 Food-101 training

Trains with the actual task 4 script (`task4_food101/train.py`), not the
toy demo above. Checkpoints go to `CHECKPOINT_DIR` on Drive (set up in
section 2), so a disconnect only costs the current epoch -- just re-run
the cell below; it auto-detects `--resume` from whether the checkpoint
already exists.

Food-101's ~5GB archive is downloaded into Colab's local, ephemeral disk
each session (faster than reading it off Drive, and cheap to redo).

`results/task4_food101.csv` (written by `log_result`) lives on Colab's
local, ephemeral disk like the Food-101 data -- it is **not** under
`CHECKPOINT_DIR`, so unlike the model checkpoints it does not survive a
disconnect on its own. The cells below copy any existing Drive copy back
into the repo before each run (so rows from earlier sessions aren't lost)
and copy the updated file back to Drive right after, so the results
history persists across sessions just like the checkpoints do.

In [ ]:
import shutil

LOCAL_RESULTS_CSV = f"{REPO_DIR}/results/task4_food101.csv"
DRIVE_RESULTS_CSV = f"{CHECKPOINT_DIR}/task4_food101_results.csv"


def pull_results_from_drive():
    if os.path.exists(DRIVE_RESULTS_CSV):
        os.makedirs(os.path.dirname(LOCAL_RESULTS_CSV), exist_ok=True)
        shutil.copy(DRIVE_RESULTS_CSV, LOCAL_RESULTS_CSV)


def push_results_to_drive():
    if os.path.exists(LOCAL_RESULTS_CSV):
        shutil.copy(LOCAL_RESULTS_CSV, DRIVE_RESULTS_CSV)
    else:
        print(f"WARNING: {LOCAL_RESULTS_CSV} not found, nothing to sync to Drive")

In [ ]:
import os

DATA_DIR = f"{REPO_DIR}/task4_food101/data"
TEACHER_CKPT = f"{CHECKPOINT_DIR}/task4_teacher.pt"

pull_results_from_drive()
resume_flag = "--resume" if os.path.exists(TEACHER_CKPT) else ""
!cd {REPO_DIR} && python -m task4_food101.train \
    --mode teacher --data-dir {DATA_DIR} --checkpoint {TEACHER_CKPT} {resume_flag}
push_results_to_drive()

### Distill into the student model

Run after the teacher checkpoint above exists (needs `--teacher-checkpoint`).

In [ ]:
STUDENT_CKPT = f"{CHECKPOINT_DIR}/task4_student.pt"

pull_results_from_drive()
resume_flag = "--resume" if os.path.exists(STUDENT_CKPT) else ""
!cd {REPO_DIR} && python -m task4_food101.train \
    --mode student --data-dir {DATA_DIR} \
    --teacher-checkpoint {TEACHER_CKPT} --checkpoint {STUDENT_CKPT} {resume_flag}
push_results_to_drive()

### Ablation: student without distillation

Optional, but needed to show distillation (not just a smaller
architecture) is responsible for the student's accuracy: same
MobileNetV3 student, same hyperparameters, but `--distillation-weight 0`
ignores the teacher's soft targets and trains on hard labels only. Uses
a separate checkpoint path so it doesn't overwrite the distilled student.

In [ ]:
STUDENT_NO_DISTILL_CKPT = f"{CHECKPOINT_DIR}/task4_student_no_distill.pt"

pull_results_from_drive()
resume_flag = "--resume" if os.path.exists(STUDENT_NO_DISTILL_CKPT) else ""
!cd {REPO_DIR} && python -m task4_food101.train \
    --mode student --data-dir {DATA_DIR} \
    --teacher-checkpoint {TEACHER_CKPT} --checkpoint {STUDENT_NO_DISTILL_CKPT} \
    --distillation-weight 0 {resume_flag}
push_results_to_drive()

## 7. Food-101 compression study

Run sections 1-3, then the cells in this section in order. Section 6 is the older ResNet-50 experiment and can be skipped. **Push these new project files to the branch Colab clones, or otherwise copy them into the Colab checkout first**; local uncommitted changes are not transferred by `git clone`. The older Drive checkpoints (`task4_teacher.pt` and `task4_student.pt`) use a different architecture/pipeline and cannot be loaded here.

This is a substantial sweep, not a seven-minute training job. Checkpoints, logit cache, CSV and figures are stored in Drive. Training resumes at epoch boundaries. Run all measurements again in one Colab session on the same CPU (Colab hardware can change after a disconnect). The maximum allowed drop is fixed at **2 percentage points** before the runs. See the README for methodology, reference-data limitations, and the final selection rule.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
from torchvision.datasets import Food101

repo = Path(REPO_DIR)
if not (repo / 'task4_food101/experiment.py').is_file():
    raise FileNotFoundError('The Colab checkout is missing the new task 4 files; push or copy them first.')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Colab before running the training study.')
runs = Path(CHECKPOINT_DIR) / 'food101_compression'
runs.mkdir(parents=True, exist_ok=True)
food_data = repo / 'task4_food101/data'
results_csv = runs / 'comparison.csv'
logits_path = runs / 'swin_logits.pt'
Food101(root=food_data, split='train', download=True)

def experiment(*args):
    cmd = [sys.executable, '-u', '-m', 'task4_food101.experiment', *map(str, args)]
    with subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
        status = proc.wait()
    if status:
        raise subprocess.CalledProcessError(status, cmd)

def train_run(output, *options):
    resume = output.with_name(output.name + '.resume.pt')
    if output.exists() and not resume.exists():
        raise RuntimeError(f'{output} exists without its resume checkpoint; use a new run directory.')
    experiment('train', '--data-dir', food_data, '--output', output, *options, *(['--resume'] if resume.exists() else []))

def measure_run(name, split, path=None):
    source = ['--reference'] if path is None else ['--input', path]
    experiment('measure', '--data-dir', food_data, '--results', results_csv, '--batch-size', 8,
               '--threads', 1, '--name', name, '--split', split, *source)

print('Study directory:', runs)

### Reference, cache and student sweep

The reference test measurement fixes the threshold. Only student validation is measured during tuning. After a disconnect, rerun setup and this cell; completed training runs resume as no-ops. The logit cache is validated against the split when used.

In [ ]:
measure_run('swin', 'test')
if not logits_path.exists():
    experiment('cache', '--data-dir', food_data, '--batch-size', 8, '--output', logits_path)

students = {}
for architecture in ('mobilenet_v3_large', 'efficientnet_b0', 'resnet18'):
    name = f'{architecture}_baseline'
    path = runs / f'{name}.pt'
    train_run(path, '--architecture', architecture)
    students[name] = path
    for temperature in (2, 4):
        for alpha in (0.25, 0.75):
            name = f'{architecture}_t{temperature}_a{alpha}'
            path = runs / f'{name}.pt'
            train_run(path, '--architecture', architecture, '--cache', logits_path,
                      '--temperature', temperature, '--alpha', alpha)
            students[name] = path

for name, path in students.items():
    measure_run(name, 'validation', path)

### Pruning and quantization

Select the best ResNet-18 distillation configuration by validation only, then compare pruning rates and dynamic/static/QAT quantization of both baseline and distilled ResNet-18. QAT uses hard-label fine-tuning. Existing compressed outputs are reused only when their source checkpoint still matches.

In [ ]:
import pandas as pd
from task4_food101.experiment import file_hash

scores = pd.read_csv(results_csv)
resnet_scores = scores[(scores['split'] == 'validation') & scores['name'].str.startswith('resnet18_t')]
best_name = resnet_scores.sort_values(['top1', 'name'], ascending=[False, True]).iloc[0]['name']
best = students[best_name]
print('ResNet distillation selected for compression:', best_name)
compressed = {}
for amount in (0.3, 0.5, 0.7):
    name = f'resnet18_pruned_{amount}'
    path = runs / f'{name}.pt'
    train_run(path, '--architecture', 'resnet18', '--start', best, '--prune', amount)
    compressed[name] = path

for source_name, source in [('baseline', students['resnet18_baseline']), ('distilled', best)]:
    for method in ('dynamic', 'static', 'qat'):
        name = f'resnet18_{source_name}_{method}'
        path = runs / f'{name}.pt'
        if path.exists():
            metadata = torch.load(path, map_location='cpu', weights_only=True)['metadata']
            if metadata.get('source_artifact') != file_hash(source) or metadata.get('method') != method:
                raise ValueError(f'{path} was made from another checkpoint; use a new run directory.')
        else:
            experiment('compress', '--data-dir', food_data, '--input', source, '--output', path, '--method', method)
        compressed[name] = path

for name, path in compressed.items():
    measure_run(name, 'validation', path)

### Final test and Pareto figures

Run only after configurations are frozen. The report selects the smallest model using validation, then reports whether it passes the reference-minus-two-points threshold on test. Test scores are plotted but are not used to select the model. If the runtime changed, rerun both validation measurement loops and the reference measurement first.

In [ ]:
for name, path in {**students, **compressed}.items():
    measure_run(name, 'test', path)
subprocess.run([sys.executable, '-m', 'task4_food101.report', '--results', str(results_csv),
                '--output', str(runs)], cwd=repo, check=True)
print('Results and figures saved in', runs)